# Phase 3 — AI-Powered Insight Layer
Integrates the Anthropic Claude API to generate analyst-style commentary on model performance and explain prediction divergence events exceeding a 2% threshold.

## Setup

In [45]:
import anthropic
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display
import pandas as pd
import numpy as np
load_dotenv()

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_KEY"))

print("Client ready.")

Client ready.


In [46]:
USE_CACHED = True  # Set to False to re-run API calls and overwrite saved results

In [47]:
import pickle

with open("../data/train_test_data.pkl", "rb") as f:
    data =pickle.load(f)

import pandas as pd
comparison_df = pd.read_csv("../outputs/model_comparison.csv", index_col= "Model")

print(comparison_df)

                              model       rmse       mae  directional_accuracy
Model                                                                         
LinearRegression  Linear Regression   2.067377  1.578433                 68.88
LSTM                           LSTM   7.236123  5.971854                 52.13
RandomForest           RandomForest  10.599141  8.051550                 63.90
XGBoost                     XGBoost  11.012762  8.554283                 67.22


## Overall Model Commentary

In [50]:
def get_model_commentary(comparison_df):
    """
    Input: comparsion dataframe
    output: analyst-style commentary from Claude
    """
    #dataframe to string
    metrics_str = comparison_df.to_string()

    #THE PROMPT ENGINEERING

    prompt = f"""
    You are a quantitative analyst reviewing the results of a stock price forecasting experiment.
    Four models are trained on 5 years of AAPL historical data and evaluated on three metrics:
    - RMSE (Root Mean Square Error) - lower is better
    - MAE (Mean Average Value) - lower is better
    - Directional Accuracy - higher is better, 50% is a coin flip
    - For RMSE and MAE, these measure the difference between the actual and predicted stock prices

    Here are the results:
    {metrics_str}

    Write a concise analyst-style commentary (3-4 sentences) that:
    1. Identifies the best performing model and why
    2. Notes any interesting patterns between the models
    3. Comments on what the directional accuracy results mean practically
    Keep the tone professional but accessible to a non-technical audience. 
    Key ideas should be clearly conveyed.

"""
    message = client.messages.create(
        model= "claude-opus-4-5",
        max_tokens= 500,
        messages=[
            {"role" : "user", "content": prompt}
        ]
    )
    return(message.content[0].text)


In [51]:
if USE_CACHED:
    with open("../outputs/overall_commentary.json", "r") as f:
        commentary = json.load(f)["overall_commentary"]
    print("Loaded overall commentary from cache.")
else:
    commentary = get_model_commentary(comparison_df)

Loaded overall commentary from cache.


In [52]:
display(Markdown(commentary))


## Model Performance Analysis

**Linear Regression emerges as the clear winner**, delivering the lowest prediction errors (RMSE: 2.07, MAE: 1.58) by a substantial margin—nearly four times more accurate than the next best model. Interestingly, the more complex machine learning models (LSTM, Random Forest, XGBoost) significantly underperformed the simpler Linear Regression approach, suggesting that AAPL's price movements over this period may follow relatively straightforward trends that don't require sophisticated pattern recognition.

From a practical trading perspective, the directional accuracy results are encouraging: Linear Regression correctly predicted whether the stock would move up or down nearly 69% of the time—well above the 50% random baseline. This means an investor using this model's directional signals would have been right roughly 7 out of 10 times, providing a meaningful edge for timing decisions, though the absolute price predictions should still be used cautiously given inherent market uncertainties.

## Per-Model Commentary

In [53]:
def get_per_model_commentary(model_name, metrics):
    """
    Input: model names and their metrics (MAE, RMSE and Direcrtional accuracy) as dict
    Output: analyst-style commentary for each model
    """
    prompt = f"""
    You are a seasoned quantitative analyst wrting a brief performace review for a single model
    in a stock price forecasting experiment

    Model:{model_name}
    RMSE : {metrics['rmse']} (lower is better, measures average dollar error)
    MSE : {metrics['mae']} (lower is better, average absolute dollar error)
    Directional Accuracy : {metrics['directional_accuracy']} ( higher is better and above 50% beats coin flip)

    Context: This model was one of four tested on 5 years of AAPL data.
    The best performing model achieved an RMSE of 2.07, MAE of 1.58, and 68.88% directional accuracy.

    Write 2-3 sentences that:
    1. Summarize how this model perfomed
    2. Compare is to the benchmarks above
    3. Give one practical insight about what these numbers mean

    Be direct and professional. No bullet points23, just prose
"""
    
    message = client.messages.create(
        model = 'claude-opus-4-5',
        max_tokens= 500,
        messages= [
            {"role" : "user", "content" : prompt}
        ]
    )
    return(message.content[0].text)

In [54]:
if USE_CACHED:
    with open("../outputs/per_model_commentary.json", "r") as f:
        per_model_commentary = json.load(f)
    print("Loaded per-model commentary from cache.")
else:
    per_model_commentary = {}

    for model_name, row in comparison_df.iterrows():
        print(f'Genterating commentary for {model_name}..')

        metrics = {
            "rmse" : row["rmse"],
            "mae" : row["mae"],
            "directional_accuracy" : row["directional_accuracy"]
        }
        model_commentary = get_per_model_commentary(model_name, metrics)
        per_model_commentary[model_name] = model_commentary
        print(f"Done.\n")

    print(f'All commentaries generated')

Loaded per-model commentary from cache.


In [56]:
with open("../outputs/per_model_commentary.json","w") as f:
    json.dump(per_model_commentary, f , indent= 4)

print("Per-model commentary saved to outputs/per_model_commentary.json")

overall ={"overall_commentary": commentary}

with open("../outputs/overall_commentary.json", "w") as f:
    json.dump(overall, f, indent= 4)

print("Overall commentary of the models saved to outputs/overall_commentary.json")

Per-model commentary saved to outputs/per_model_commentary.json
Overall commentary of the models saved to outputs/overall_commentary.json


In [57]:
with open("../outputs/per_model_commentary.json", "r") as f:
    saved = json.load(f)

for model, text in saved.items():
    print(f"\n {'='*50}")
    print(f"Model: {model}")
    print(f"f{'='*50}")
    print(text)


Model: LinearRegression
f==================================================
**Performance Review: Linear Regression Model**

The Linear Regression model delivered strong performance on the AAPL forecasting task, achieving an RMSE of $2.07, MAE of $1.58, and directional accuracy of 68.88%, effectively matching the best benchmarks across all three metrics. This indicates that despite its relative simplicity, the model captures the underlying price dynamics as effectively as more complex alternatives tested in this experiment. From a practical standpoint, the 68.88% directional accuracy suggests the model correctly predicts whether AAPL will move up or down roughly 7 out of 10 trading days, providing meaningful edge over random chance for directional trading strategies.

Model: LSTM
f==================================================
**Performance Review: LSTM Model**

The LSTM model delivered underwhelming results with an RMSE of $8.02 and directional accuracy of 54.03%, placing it well

## Divergence Detection

In [ ]:
def detect_divergences(y_test, y_pred, dates, threshold_pct=0.02):
    """
    Flags dates where predicted vs actual price diverges 
    beyond a percentage threshold.
    
    threshold_pct=0.02 means flag anything over 2% off
    """
    
    y_test_arr = np.array(y_test).flatten()
    y_pred_arr = np.array(y_pred).flatten()
    dates_arr = np.array(dates)
    # Calculate percentage divergence for each prediction
    pct_divergence = np.abs(y_test_arr - y_pred_arr) / y_test_arr
    
    # Flag anything beyond the threshold
    divergence_mask = pct_divergence > threshold_pct
    
    # Build a DataFrame of flagged dates
    divergence_df = pd.DataFrame({
        "date":           dates[divergence_mask],
        "actual":         y_test_arr[divergence_mask].round(2),
        "predicted":      y_pred_arr[divergence_mask].round(2),
        "pct_divergence": (pct_divergence[divergence_mask] * 100).round(2)
    })
    
    divergence_df = divergence_df.sort_values("pct_divergence", ascending=False)
    divergence_df = divergence_df.reset_index(drop=True)
    
    return divergence_df

In [ ]:
dates = pd.Series(data['y_test'].index).reset_index(drop=True)

divergence_df = detect_divergences(
    y_test= data['y_test'],
    y_pred= data['model_predictions']['LinearRegression'],
    dates = dates,
    threshold_pct= 0.02
)

print(f"Total divergences detected: {len(divergence_df)}")
print(divergence_df.head(10))

Total divergences detected: 25
         date  actual  predicted  pct_divergence
15 2023-11-01  171.95     167.78            2.43
16 2023-08-23  179.02     174.67            2.43
17 2023-09-01  187.27     183.08            2.24
18 2023-11-07  179.71     175.78            2.19
19 2023-10-17  175.10     178.88            2.16
20 2023-02-03  152.06     148.83            2.13
21 2023-03-20  155.15     151.90            2.09
22 2023-03-03  148.87     145.78            2.08
23 2023-10-09  176.92     173.25            2.07
24 2023-10-06  175.43     171.81            2.06


In [60]:
divergence_df.to_csv("../outputs/divergence_events.csv", index = False)
print("Divergence events saved")

Divergence events saved


## AI-Powered Divergence Explanation

In [61]:
def explain_divergences(date, actual, predicted, pct_divergence):
    """
    Analyze divergence events one at a time
    and Claude suggests reasoning
    """

    prompt =f"""
    You are a quantitative analyst reviewing anomalies stock price
    forecasting model trained on AAPL data.

    On {date}, the model produced a significant prediction error:
    - Actual Price:     ${actual}
    - Predicted Price:  ${predicted}
    -Divergence:        {pct_divergence}%

    In 2-3 sentence, suggest the most likely real-world cause for 
    this prediction error. Consider: earnings announcements, Fed decisions
    broader market volatility, macro events, or sector-specific news. Be 
    specific to the date possible. Keep the tone analytical and concise.
"""
    message = client.messages.create(
        model= 'claude-opus-4-5',
        max_tokens= 250,
        messages=[
            {"role" : "user", "content": prompt}
        ]
    )
    return(message.content[0].text)

In [62]:
divergence_insights={}
top_divergences =divergence_df.head()

for _, row in top_divergences.iterrows():
    date_str = str(row["date"])[:10]
    print(f"Analyzing divergence on {date_str}...")

    explanation = explain_divergences(
        date = date_str,
        actual = row["actual"],
        predicted = row["predicted"],
        pct_divergence = row["pct_divergence"]
    )

    divergence_insights[date_str] = {
        "actual" :          row["actual"],
        "predicted" :       row["predicted"],
        "pct_divergence" :  row["pct_divergence"],
        "explanation" :     explanation
    }

    print(f"Done. \n")

print(f"All divergence explanations generated")



Analyzing divergence on 2023-08-04...
Done. 

Analyzing divergence on 2023-08-07...
Done. 

Analyzing divergence on 2023-09-07...
Done. 

Analyzing divergence on 2023-09-08...
Done. 

Analyzing divergence on 2023-03-08...
Done. 

All divergence explanations generated


In [63]:
with open("../outputs/divergence_insights.json", "w") as f:
    json.dump(divergence_insights, f, indent= 4)

print("Divergence insights saved.")

Divergence insights saved.


In [64]:
with open("../outputs/divergence_insights.json", "r") as f:
    saved_insights = json.load(f)

for date, insight in saved_insights.items():
    print(f"\n{'='*55}")
    print(f"Date:       {date}")
    print(f"Actual:     ${insight['actual']}")
    print(f"Predicted:  ${insight['predicted']}")
    print(f"Divergence: {insight['pct_divergence']}")
    print(f"{'='*55}")
    print(insight['explanation'])


Date:       2023-08-04
Actual:     $179.64
Predicted:  $186.78
Divergence: 3.98
## Analysis of AAPL Prediction Error on August 4, 2023

The prediction error most likely stems from the aftershock of Apple's Q3 FY2023 earnings report released on **August 3, 2023**, which showed the third consecutive quarter of revenue decline and weaker-than-expected iPhone sales, causing shares to drop approximately 4.8% on August 4th. The model likely failed to adequately price in the negative forward guidance on Q4 revenue, as management projected another year-over-year decline, which conflicted with the model's presumably bullish momentum signals heading into earnings. Additionally, this occurred during a period of broader tech sector pressure following Fitch's U.S. credit rating downgrade on August 1st, compounding the negative sentiment.

Date:       2023-08-07
Actual:     $176.54
Predicted:  $182.76
Divergence: 3.53
## Analysis of AAPL Prediction Error on August 7, 2023

The prediction error like